# FEATURE ENGINEERING
## Engine Health Monitoring – CMAPSS Dataset

### Objective
Transform raw sensor data into machine-learning ready features 
for Remaining Useful Life (RUL) prediction.

### **Import Libraries**

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

### **Load Processed Dataset**

In [2]:
DATA_PATH = "../data/processed/train_engine_data_clean.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


### **STEP 1: Generate RUL**

RUL = Remaining Useful Life

RUL = max_cycle_of_engine - current_cycle

### **Calculate Max Cycle Per Engine**

In [3]:
max_cycle = df.groupby("engine_id")["cycle"].max().reset_index()
max_cycle.columns = ["engine_id", "max_cycle"]

max_cycle.head()

,engine_id,max_cycle
0,1,192
1,2,287
2,3,179
3,4,189
4,5,269


### **Merge Back to Dataset**

In [4]:
df = df.merge(max_cycle, on="engine_id", how="left")

df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,max_cycle
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,192
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,192
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,192
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,192
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,192


### **Create RUL Column**

In [5]:
df["RUL"] = df["max_cycle"] - df["cycle"]

df[["engine_id", "cycle", "max_cycle", "RUL"]].head()

,engine_id,cycle,max_cycle,RUL
0,1,1,192,191
1,1,2,192,190
2,1,3,192,189
3,1,4,192,188
4,1,5,192,187


### **Remove Helper Column**

In [6]:
df.drop(columns=["max_cycle"], inplace=True)

df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


### **STEP 2: Cap RUL**

Why?

Early cycles have huge RUL (300+).
Models don’t benefit from very large values.

Standard NASA practice: cap at 125.

### **Apply RUL Cap**

In [7]:
RUL_CAP = 125

df["RUL"] = df["RUL"].clip(upper=RUL_CAP)

df["RUL"].describe()

count    20631.000000
mean        86.829286
std         41.673699
min          0.000000
25%         51.000000
50%        103.000000
75%        125.000000
max        125.000000
Name: RUL, dtype: float64

### **STEP 3: Remove Low-Variance Sensors**

### **Drop Low-Variance Sensors**

In [8]:
low_variance_sensors = [
    "sensor_1",
    "sensor_5",
    "sensor_6",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

df.drop(columns=low_variance_sensors, inplace=True)

df.shape

(20631, 20)

### **STEP 4: Drop Identifier Columns from Features**

### **Separate Features and Target**

In [9]:
X = df.drop(columns=["engine_id", "cycle", "RUL"])
y = df["RUL"]

X.head()

,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
0,-0.0007,-0.0004,100.0,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190
1,0.0019,-0.0003,100.0,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236
2,-0.0043,0.0003,100.0,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442
3,0.0007,0.0000,100.0,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739
4,-0.0019,-0.0002,100.0,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044


### **STEP 5: Rolling Mean Features**

### **Add Rolling Mean (Window=5)**

In [10]:
sensor_cols = [col for col in X.columns if "sensor" in col]

for col in sensor_cols:
    df[f"{col}_roll5"] = (
        df.groupby("engine_id")[col]
        .rolling(5)
        .mean()
        .reset_index(0, drop=True)
    )

df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_8_roll5,sensor_9_roll5,sensor_11_roll5,sensor_12_roll5,sensor_13_roll5,sensor_14_roll5,sensor_15_roll5,sensor_17_roll5,sensor_20_roll5,sensor_21_roll5
0,1,1,-0.0007,-0.0004,100.0,641.82,1589.70,1400.60,554.36,2388.06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,0.0019,-0.0003,100.0,642.15,1591.82,1403.14,553.75,2388.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,-0.0043,0.0003,100.0,642.35,1587.99,1404.20,554.26,2388.08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,4,0.0007,0.0000,100.0,642.35,1582.79,1401.87,554.45,2388.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,5,-0.0019,-0.0002,100.0,642.37,1582.85,1406.22,554.00,2388.06,...,2388.07,9049.566,47.328,522.282,2388.048,8134.194,8.41334,391.8,38.958,23.39302


### **Fill Initial NaNs**

In [11]:
df.fillna(method="bfill", inplace=True)

In [12]:
df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_8_roll5,sensor_9_roll5,sensor_11_roll5,sensor_12_roll5,sensor_13_roll5,sensor_14_roll5,sensor_15_roll5,sensor_17_roll5,sensor_20_roll5,sensor_21_roll5
0,1,1,-0.0007,-0.0004,100.0,641.82,1589.70,1400.60,554.36,2388.06,...,2388.07,9049.566,47.328,522.282,2388.048,8134.194,8.41334,391.8,38.958,23.39302
1,1,2,0.0019,-0.0003,100.0,642.15,1591.82,1403.14,553.75,2388.04,...,2388.07,9049.566,47.328,522.282,2388.048,8134.194,8.41334,391.8,38.958,23.39302
2,1,3,-0.0043,0.0003,100.0,642.35,1587.99,1404.20,554.26,2388.08,...,2388.07,9049.566,47.328,522.282,2388.048,8134.194,8.41334,391.8,38.958,23.39302
3,1,4,0.0007,0.0000,100.0,642.35,1582.79,1401.87,554.45,2388.11,...,2388.07,9049.566,47.328,522.282,2388.048,8134.194,8.41334,391.8,38.958,23.39302
4,1,5,-0.0019,-0.0002,100.0,642.37,1582.85,1406.22,554.00,2388.06,...,2388.07,9049.566,47.328,522.282,2388.048,8134.194,8.41334,391.8,38.958,23.39302


### **STEP 6: Feature Scaling**

### **Scale Features**

In [13]:
feature_cols = [col for col in df.columns if col not in ["engine_id", "cycle", "RUL"]]

scaler = StandardScaler()

df[feature_cols] = scaler.fit_transform(df[feature_cols])

df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_8_roll5,sensor_9_roll5,sensor_11_roll5,sensor_12_roll5,sensor_13_roll5,sensor_14_roll5,sensor_15_roll5,sensor_17_roll5,sensor_20_roll5,sensor_21_roll5
0,1,1,-0.315980,-1.372953,0.0,-1.721725,-0.134255,-0.925936,1.121141,-0.516338,...,-0.389199,-0.732906,-0.83956,1.266371,-0.715037,-0.512732,-0.867455,-1.088097,0.898228,1.09758
1,1,2,0.872722,-1.031720,0.0,-1.061780,0.211528,-0.643726,0.431930,-0.798093,...,-0.389199,-0.732906,-0.83956,1.266371,-0.715037,-0.512732,-0.867455,-1.088097,0.898228,1.09758
2,1,3,-1.961874,1.015677,0.0,-0.661813,-0.413166,-0.525953,1.008155,-0.234584,...,-0.389199,-0.732906,-0.83956,1.266371,-0.715037,-0.512732,-0.867455,-1.088097,0.898228,1.09758
3,1,4,0.324090,-0.008022,0.0,-0.661813,-1.261314,-0.784831,1.222827,0.188048,...,-0.389199,-0.732906,-0.83956,1.266371,-0.715037,-0.512732,-0.867455,-1.088097,0.898228,1.09758
4,1,5,-0.864611,-0.690488,0.0,-0.621816,-1.251528,-0.301518,0.714393,-0.516338,...,-0.389199,-0.732906,-0.83956,1.266371,-0.715037,-0.512732,-0.867455,-1.088097,0.898228,1.09758


### **STEP 7: Final Dataset Split**

### **Train/Test Split by Engine**

In [14]:
engine_ids = df["engine_id"].unique()

train_engines = engine_ids[:80]
test_engines = engine_ids[80:]

train_df = df[df["engine_id"].isin(train_engines)]
test_df = df[df["engine_id"].isin(test_engines)]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16138, 34)
Test shape: (4493, 34)


### **STEP 8: Save Final Dataset**

### **Save Engineered Data**

In [15]:
train_df.to_csv("../data/processed/train_engineered.csv", index=False)
test_df.to_csv("../data/processed/test_engineered.csv", index=False)